In [ ]:
# Colab setup -- installs SoftMobility when running on Google Colab.
# Safe to run locally: it does nothing outside Colab.
try:
    import google.colab  # noqa: F401
    %pip install -q git+https://github.com/C0PEP0D/SoftMobility.git
except ImportError:
    pass

# Jeffery's orbits of a rigid body

### Imports

In [1]:
import jax.numpy as jnp
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import softmobility as sm

np.set_printoptions(precision=3, linewidth=100, suppress=True, sign=" ")

## Assembly creation

### Body shape 

In [2]:
yaml_data = """
# Variable Prefixes (Optional)
design_names:     # Design parameter prefixes/names (extends default "design")
  - myradius      # Recognized as a parameter prefix/name (e.g., `myradius` in expressions)
  - distance
  - offset

# Default Values (Optional)
defaults:
  myradius0: 1.   # Parameter: Listed in `design_names`
  myradius1: 1.
  distance: 1.

# Spheres (Mandatory)
spheres:
  - # Sphere 0 #################
    radius: myradius0
    position: [offset - myradius0 - distance/2, 0, 0]

  - # Sphere 1 #################
    radius: myradius1
    position: [offset + myradius1 + distance/2, 0, 0]
"""

In [3]:
mybody= sm.SoftBody(yaml_data, verbose=False)
print(repr(mybody))

SPHERE ASSEMBLY
  2 spheres
  0 degrees of freedom
  4 design parameters
  0 input parameters

Default values
  degrees of freedom dof: [] = []
  design parameters param: ['distance', 'myradius0', 'myradius1', 'offset'] = [ 1.  1.  1.  0.]
  input parameters param: []

SPHERE 0
  radius: 1.0
  position: [-1.5  0.   0. ]
  orientation: [ 0.  0.  0.]
  c_field:
[]
  c_stiff:
[]

SPHERE 1
  radius: 1.0
  position: [ 1.5  0.   0. ]
  orientation: [ 0.  0.  0.]
  c_field:
[]
  c_stiff:
[]



## Fluid-structure interaction problem

In [4]:
# Create a shear flow with shear rate 1
shear_flow = sm.shear_flow(shear_rate=1.0)

# Test it
pos = [1.0, 2.0, 3.0]
grad_u = shear_flow.gradient(pos)
Omega, rate_of_strain = shear_flow.omega_rate_of_strain(pos)
print("Velocity Gradient Tensor (∇u):\n", grad_u)
print("Angular velocity Ω:", Omega)
print("Rate-of-strain E):\n", rate_of_strain)

Velocity Gradient Tensor (∇u):
 [[ 0.  1.  0.]
 [ 0.  0.  0.]
 [ 0.  0.  0.]]
Angular velocity Ω: [ 0.   0.  -0.5]
Rate-of-strain E):
 [[ 0.   0.5  0. ]
 [ 0.5  0.   0. ]
 [ 0.   0.   0. ]]


### Numerics

In [5]:
# parameters
final_time = 30
dt         = 0.02
n_steps    = int(final_time / dt)

# simulation
rollout = sm.FlowBodyRollout(soft_body=mybody, flow=shear_flow)
positions, orientations, dofs = rollout.rollout(dt=dt, n_steps=n_steps, init_orientation=jnp.array([0., 0.6, 0.]))

### Figure setup

In [6]:
# Time vector
t = jnp.linspace(dt, final_time, n_steps)  # starts at dt since rollout output is post-step

# Create a subplot figure with 3 rows and 1 column
fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=("DOF", "Position (X, Y, Z)", "Orientation (X, Y, Z)"),
    shared_xaxes=True,  # Share the x-axis for all plots (time)
    vertical_spacing=0.1  # Adjust vertical spacing (reduce clutter)
)

# Plot position components (X, Y, Z) in the second subplot
fig.add_trace(go.Scatter(x=t, y=positions[:, 0], mode='lines', name='Position X'), row=1, col=1)
fig.add_trace(go.Scatter(x=t, y=positions[:, 1], mode='lines', name='Position Y'), row=1, col=1)
fig.add_trace(go.Scatter(x=t, y=positions[:, 2], mode='lines', name='Position Z'), row=1, col=1)

# Plot orientation components (X, Y, Z) in the third subplot
fig.add_trace(go.Scatter(x=t, y=orientations[:, 0], mode='lines', name='Orientation X'), row=2, col=1)
fig.add_trace(go.Scatter(x=t, y=orientations[:, 1], mode='lines', name='Orientation Y'), row=2, col=1)
fig.add_trace(go.Scatter(x=t, y=orientations[:, 2], mode='lines', name='Orientation Z'), row=2, col=1)

# Update layout for titles and labels
axis_style = dict(
    showline=True, linewidth=1, linecolor="black", mirror=True,
    showgrid=False, zeroline=False,
    ticks="inside", tickwidth=1, tickcolor="black",
)

fig.update_layout(
    title=dict(text="Trajectory Over Time", font=dict(family="serif", size=14)),
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(family="Arial", size=16),
    height=800, width=800,
    legend=dict(bordercolor="black", borderwidth=1, bgcolor="white"),
    # Hide x tick labels on top subplot, show only on bottom
    xaxis=dict(**axis_style, showticklabels=False),
    xaxis2=dict(**axis_style, title="t"),
    # Position: free y axis
    yaxis=dict(**axis_style, title="Position"),
    # Orientation: angle ticks
    yaxis2=dict(
        **axis_style,
        title="Orientation",
        tickvals=[-jnp.pi, -jnp.pi/2, 0, jnp.pi/2, jnp.pi],
        ticktext=["-π", "-π/2", "0", "π/2", "π"],
    ),
)

### Figure

In [7]:
fig.show()

### Theory

In [ ]:
phi_num = np.zeros_like(t)
theta_num = np.zeros_like(t)
p0 = np.array([1, 0, 0])

for i, _ti in enumerate(t):
    R = np.asarray(sm.rotation_matrix(orientations[i]))
    p = R @ p0
    phi_num[i] = np.atan2(p[1], p[0])
    theta_num[i] = np.atan(np.sqrt((1 - p[1]**2 - p[0]**2) / (p[1]**2 + p[0]**2)))

In [10]:
tensors = mybody.compute_tensors()
M0 = tensors.M @ mybody.compute_composition_of_forces()  # Mobility with forces/torques at the ref point x_0
N = mybody.Nspheres
# Reshape M into a 3D array where each element is a 6x6 matrix representing one sphere's data
M_blocks = M0.reshape(6, 6 * N).T.reshape(N, 6, 6)
M_mean = np.mean(M_blocks, axis=0)

# Bretherton parameter
Bretherton = tensors.C_E[-1,1] # * M_mean[1,1]

# print("Rigid mobility tensor\n", M_mean)
# print("Coupling with strain\n", tensors.C_E)    # Exx, Exy, Exz, Eyy, Eyz
print("Bretherton parameters:", Bretherton)

Bretherton parameters: 0.7168667


In [11]:
time = np.linspace(0,t[-1],num=100)
phi0 = phi_num[0]
theta0 = theta_num[0]
c = np.sqrt((1 + Bretherton) / (1 - Bretherton))
K = np.sqrt(np.cos(phi0)**2 + c**2 * np.sin(phi0)) / np.tan(theta0)

phi = np.atan2(-(1/c) * np.sin(time / (c + 1/c)), np.cos(time / (c + 1/c)))
# phi = np.atan(-(1/c) * np.tan(time / (c + 1/c)))
theta = np.atan(np.sqrt(np.cos(phi)**2 + c**2 * np.sin(phi)**2) / K)

### Figure setup

In [12]:
fig = go.Figure()

fig.add_trace(go.Scatter(x=t, y=phi_num, mode='lines', name='Numerics phi'))
fig.add_trace(go.Scatter(x=t, y=theta_num, mode='lines', name='Numerics theta'))
fig.add_trace(go.Scatter(x=time, y=phi, mode='markers', name='Theory phi'))
fig.add_trace(go.Scatter(x=time, y=theta, mode='markers', name='Theory theta'))

fig.update_layout(
    title=dict(text="Comparison of Jeffery's orbits", font=dict(size=14)),
    height=600,  # Set figure height
    width=800,   # Set figure width
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(family="Arial", size=16, color="black"),
    xaxis=dict(
        title="t",
        showline=True, linewidth=1, linecolor="black", mirror=True,  # axis box
        showgrid=False, zeroline=False,
        ticks="inside", tickwidth=1, tickcolor="black",
    ),
    yaxis=dict(
        title="φ, θ",
        tickvals=[-np.pi, -np.pi/2, 0, np.pi/2, np.pi],
        ticktext=["-π", "-π/2", "0", "π/2", "π"],
        showline=True, linewidth=1, linecolor="black", mirror=True,
        showgrid=False, zeroline=False,
        ticks="inside", tickwidth=1, tickcolor="black",
    ),
    legend=dict(
        bordercolor="black", borderwidth=1,
        bgcolor="white",
    ),
)

### Figure

In [13]:
fig.show()